# Public release note

This is a sanitized public copy of `omniASR_20_K5_KenLM_V6_edge_audit(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# 20.K5 — Audit edge/RSS du candidat KenLM V6

Ce notebook vérifie la configuration issue de **20.K4** avant toute inférence test. Il mesure le pic RAM dans un sous-processus CPU propre, exerce les 12 routes de décodage avec un seul KenLM résident, puis publie un package candidat immuable.

**Il ne modifie pas la configuration active et ne crée aucune soumission.**

## Ordre d'exécution

1. Exécuter **20.K5.A**. Si Colab demande un redémarrage, redémarrer puis rejouer A.
2. Exécuter **20.K5.B** une seule fois et attendre le statut final.
3. Exécuter **20.K5.C** pour relire le résultat enregistré.

CPU + RAM augmentée suffisent. Durée typique : **5 à 20 minutes**, hors première installation.

In [ ]:
# 20.K5.A — Setup Colab idempotent (CPU + RAM augmentée suffisent)
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata as md
import json, os, subprocess, sys
from pathlib import Path

loaded_before = {name for name in ("torch", "numpy") if name in sys.modules}

def version_or_none(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

before = {name: version_or_none(name) for name in ("torch", "numpy")}

# Reprendre exactement le commit logiciel figé par 20.K4.
k4_root = Path("/content/afrivoices_project/finetune_runs/experiment_run/reports/kenlm_v6_20_K4")
k4_latest = json.load(open(k4_root / "LATEST_PASS.json", encoding="utf-8"))
assert k4_latest["run_id"] == "REPLACE_WITH_LOCAL_RUN_ID"
k4_contract = json.load(open(Path(k4_latest["report"]).parent / "contract.json", encoding="utf-8"))
commit = k4_contract["contract"]["software_fingerprint"]["omnilingual_asr_git_commit"]
assert len(commit) == 40

repo = "/content/omnilingual-asr-k5"
if not os.path.isdir(os.path.join(repo, ".git")):
    subprocess.run([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/facebookresearch/omnilingual-asr.git", repo,
    ], check=True)
subprocess.run(["git", "-C", repo, "fetch", "--depth", "1", "origin", commit], check=True)
subprocess.run(["git", "-C", repo, "checkout", "--detach", commit], check=True)

requirements = [
    repo,
    "pyarrow>=20.0.0",
    "pandas>=2.2.0",
    "numpy<3",
    "soundfile>=0.12.1",
    "pyctcdecode>=0.5.0",
    "kenlm>=0.3.0",
    "psutil>=5.9.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)

# Aligner les extensions binaires Colab sur la version de torch imposée par fairseq2.
torch_base = md.version("torch").split("+")[0]
torch_mm = ".".join(torch_base.split(".")[:2])
tv_for_torch = {
    "2.8": "0.23.0", "2.9": "0.24.0", "2.10": "0.25.0", "2.11": "0.26.0"
}
targets = {"torchaudio": torch_base, "torchvision": tv_for_torch.get(torch_mm)}
realigned = False
for package, target in targets.items():
    if not target:
        continue
    current = version_or_none(package)
    if current and current.split("+")[0] != target:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"{package}=={target}"],
            check=True,
        )
        realigned = True

after = {name: version_or_none(name) for name in ("torch", "numpy")}
changed_loaded = [
    name for name in loaded_before
    if before[name] and after[name] and before[name].split("+")[0] != after[name].split("+")[0]
]
if changed_loaded or realigned:
    print("⚠️ Des bibliothèques binaires ont été réalignées.")
    print("Runtime > Redémarrer la session, puis réexécuter 20.K5.A et 20.K5.B.")
    raise SystemExit("REDÉMARRAGE COLAB REQUIS")

actual_commit = subprocess.run(
    ["git", "-C", repo, "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert actual_commit == commit
print("✅ Setup 20.K5 prêt")
print("GPU requis : non | Exécution recommandée : CPU + RAM augmentée")
print("torch :", md.version("torch"), "| fairseq2 :", md.version("fairseq2"))
print("omnilingual-asr commit :", actual_commit)


## 20.K5.B — Audit edge complet

Cette cellule peut être relancée : un résultat complet est vérifié puis réutilisé. Le PASS exige `pic RSS × 1,15 < 8 Gio`, 985 573 552 paramètres, deux sondes DEV réelles et 12/12 routes de décodage.

In [ ]:
K5_WORKER_SOURCE = '"""Worker CPU propre pour l\'audit edge 20.K5.\n\nCe fichier est embarque sous forme de texte dans le notebook 20.K5. Il ne doit\njamais etre execute directement sans le JSON d\'entree prepare par 20.K5.B.\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport sys\nimport time\nfrom collections import OrderedDict\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef sha256_file(path: Path, block_size: int = 16 << 20) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(block_size), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json(value: Any, path: Path) -> None:\n    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)\n    os.replace(temporary, path)\n\n\ndef emit(stage: str, **values: Any) -> None:\n    record = {"stage": stage, "monotonic": time.monotonic(), **values}\n    print("K5_EVENT " + json.dumps(record, sort_keys=True), flush=True)\n\n\nassert len(sys.argv) == 3, "usage: worker.py INPUT_JSON RESULT_JSON"\nINPUT_PATH = Path(sys.argv[1]).resolve()\nRESULT_PATH = Path(sys.argv[2]).resolve()\nSPEC = json.loads(INPUT_PATH.read_text(encoding="utf-8"))\n\ntorch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))\ntorch.set_num_interop_threads(1)\nassert os.environ.get("CUDA_VISIBLE_DEVICES", "") == ""\nassert not torch.cuda.is_available(), "Le worker edge doit etre strictement CPU."\n\nBASE_PARAMETERS = int(SPEC["base_parameters"])\nADAPTER_PARAMETERS = int(SPEC["adapter_parameters"])\nTOTAL_PARAMETERS = int(SPEC["total_neural_parameters"])\nassert BASE_PARAMETERS + ADAPTER_PARAMETERS == TOTAL_PARAMETERS\nassert TOTAL_PARAMETERS < 1_000_000_000\n\nBASE_WEIGHT = Path(SPEC["base_weight"]).resolve()\nADAPTER_WEIGHT = Path(SPEC["adapter_weight"]).resolve()\nLORA_PLAN_PATH = Path(SPEC["lora_plan"]).resolve()\nCONFIG_PATH = Path(SPEC["runtime_config"]).resolve()\nfor path, expected in (\n    (BASE_WEIGHT, SPEC["base_weight_sha256"]),\n    (ADAPTER_WEIGHT, SPEC["adapter_weight_sha256"]),\n    (LORA_PLAN_PATH, SPEC["lora_plan_sha256"]),\n    (CONFIG_PATH, SPEC["runtime_config_sha256"]),\n):\n    assert path.is_file(), path\n    assert sha256_file(path) == expected, (path.name, "sha256")\n\nRUNTIME_CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))\nassert set(RUNTIME_CONFIG) == set(SPEC["runtime_groups"])\nassert int(SPEC["batch_size"]) == 1\n\n\ndef projection_dimensions(module: nn.Module) -> tuple[int, int] | None:\n    weight = getattr(module, "weight", None)\n    if not isinstance(weight, torch.Tensor) or weight.ndim != 2:\n        return None\n    name = type(module).__name__.lower()\n    namespace = type(module).__module__.lower()\n    if any(token in name for token in ("embedding", "embed", "conv", "norm")):\n        return None\n    if any(True for _ in module.children()):\n        return None\n    input_dim = getattr(module, "in_features", getattr(module, "input_dim", None))\n    output_dim = getattr(module, "out_features", getattr(module, "output_dim", None))\n    input_dim = int(input_dim) if input_dim is not None else int(weight.shape[1])\n    output_dim = int(output_dim) if output_dim is not None else int(weight.shape[0])\n    if tuple(weight.shape) != (output_dim, input_dim):\n        return None\n    if not (\n        isinstance(module, nn.Linear)\n        or "linear" in name\n        or "projection" in name\n        or "projection" in namespace\n    ):\n        return None\n    return input_dim, output_dim\n\n\nclass EvalLoRAProjection(nn.Module):\n    def __init__(self, base: nn.Module, rank: int):\n        super().__init__()\n        dimensions = projection_dimensions(base)\n        assert dimensions is not None\n        input_dim, output_dim = dimensions\n        self.base = base\n        self.rank = int(rank)\n        self.scaling = 1.0\n        self.enabled = False\n        self.lora_A = nn.Parameter(\n            torch.zeros(self.rank, input_dim, dtype=base.weight.dtype),\n            requires_grad=False,\n        )\n        self.lora_B = nn.Parameter(\n            torch.zeros(output_dim, self.rank, dtype=base.weight.dtype),\n            requires_grad=False,\n        )\n\n    def forward(self, inputs: torch.Tensor, *args: Any, **kwargs: Any) -> torch.Tensor:\n        output = self.base(inputs, *args, **kwargs)\n        if not self.enabled:\n            return output\n        return output + F.linear(F.linear(inputs, self.lora_A), self.lora_B) * self.scaling\n\n\ndef parent_and_child(root: nn.Module, dotted_name: str) -> tuple[nn.Module, str]:\n    parts = dotted_name.split(".")\n    parent = root\n    for part in parts[:-1]:\n        parent = getattr(parent, part)\n    return parent, parts[-1]\n\n\ndef labels_from_tokenizer(tokenizer: Any) -> list[str]:\n    tokenizer_model = getattr(tokenizer, "_model", None)\n    for method in ("index_to_token", "id_to_piece", "IdToPiece"):\n        if tokenizer_model is not None and hasattr(tokenizer_model, method):\n            labels = [\n                str(getattr(tokenizer_model, method)(index))\n                for index in range(tokenizer.vocab_info.size)\n            ]\n            labels[0] = ""\n            assert len(labels) == tokenizer.vocab_info.size\n            assert len(set(labels)) == len(labels)\n            return labels\n    raise RuntimeError("Vocabulaire tokenizer inaccessible")\n\n\nemit("worker_start", cuda=False)\n\n# La carte doit etre ecrite avant le premier import omnilingual_asr.\ncard_name = str(SPEC["edge_card_name"])\npackage_spec = importlib.util.find_spec("omnilingual_asr")\nassert package_spec is not None and package_spec.submodule_search_locations\npackage_root = Path(next(iter(package_spec.submodule_search_locations))).resolve()\ncard_dir = package_root / "cards/models"\ncard_dir.mkdir(parents=True, exist_ok=True)\ncard_path = card_dir / f"{card_name}.yaml"\ncard_payload = (\n    f"name: {card_name}\\n"\n    "base: omniASR_CTC_1B_v2\\n"\n    f"checkpoint: {json.dumps(\'file://\' + str(BASE_WEIGHT))}\\n"\n)\ncard_path.write_text(card_payload, encoding="utf-8")\n\nfrom fairseq2.data.tokenizers.hub import load_tokenizer\nfrom fairseq2.models.hub import load_model\nfrom omnilingual_asr.models.inference.pipeline import ASRInferencePipeline\nfrom pyctcdecode import build_ctcdecoder\n\nemit("base_load_start")\nstarted = time.monotonic()\nmodel = load_model(card_name, device=torch.device("cpu"), dtype=torch.bfloat16)\nmodel.eval()\nbase_parameters_loaded = sum(parameter.numel() for parameter in model.parameters())\nassert base_parameters_loaded == BASE_PARAMETERS\nbase_load_seconds = time.monotonic() - started\nemit("base_loaded", seconds=base_load_seconds, parameters=base_parameters_loaded)\n\nLORA_PLAN = pd.read_parquet(LORA_PLAN_PATH)\nassert len(LORA_PLAN) == int(SPEC["lora_plan_rows"])\nassert int(LORA_PLAN["parameters_per_adapter"].sum()) == ADAPTER_PARAMETERS\n\nwrappers: OrderedDict[str, EvalLoRAProjection] = OrderedDict()\nfor row in LORA_PLAN.sort_values("module").to_dict("records"):\n    parent, child = parent_and_child(model, str(row["module"]))\n    base = getattr(parent, child)\n    assert projection_dimensions(base) == (\n        int(row["in_features"]), int(row["out_features"])\n    )\n    wrapper = EvalLoRAProjection(base, int(row["rank"]))\n    setattr(parent, child, wrapper)\n    wrappers[str(row["module"])] = wrapper\nassert len(wrappers) == int(SPEC["lora_plan_rows"])\nassert sum(parameter.numel() for parameter in model.parameters()) == TOTAL_PARAMETERS\n\nemit("adapter_load_start")\nraw_adapter = torch.load(ADAPTER_WEIGHT, map_location="cpu", weights_only=False)\nassert raw_adapter.get("language") == "mas"\nassert int(raw_adapter.get("step", -1)) == 1250\nassert raw_adapter.get("plan_sha256") == SPEC["lora_plan_sha256"]\nadapter_state = raw_adapter.get("state_dict", raw_adapter)\nparameters = dict(model.named_parameters())\nexpected_keys = {\n    f"{module_name}.lora_A" for module_name in wrappers\n} | {\n    f"{module_name}.lora_B" for module_name in wrappers\n}\nassert set(adapter_state) == expected_keys\ncopied = 0\nwith torch.no_grad():\n    for key, value in adapter_state.items():\n        target = parameters[key]\n        assert tuple(target.shape) == tuple(value.shape)\n        target.copy_(value.to(dtype=target.dtype))\n        copied += target.numel()\nassert copied == ADAPTER_PARAMETERS\ndel raw_adapter, adapter_state, parameters\ngc.collect()\nemit("adapter_loaded", parameters=copied, wrappers=len(wrappers))\n\ntokenizer = load_tokenizer(card_name)\nlabels = labels_from_tokenizer(tokenizer)\nlabels_sha256 = hashlib.sha256("\\0".join(labels).encode("utf-8")).hexdigest()\npipe = ASRInferencePipeline(None, model=model, tokenizer=tokenizer)\n\n\ndef set_adapter_enabled(enabled: bool) -> None:\n    for wrapper in wrappers.values():\n        wrapper.enabled = bool(enabled)\n    assert all(wrapper.enabled is bool(enabled) for wrapper in wrappers.values())\n\n\ndef capture_logits(audio_path: Path, omni_code: str, enabled: bool) -> tuple[Any, dict[str, Any]]:\n    assert audio_path.is_file()\n    set_adapter_enabled(enabled)\n    captured: list[torch.Tensor] = []\n    original_forward = pipe.model.forward\n\n    def spy(source_seqs: Any, source_seq_lens: Any, *args: Any, **kwargs: Any) -> Any:\n        output = original_forward(source_seqs, source_seq_lens, *args, **kwargs)\n        logits, layout = output[0], output[1]\n        length = layout.seq_lens[0]\n        length = int(length.item() if hasattr(length, "item") else length)\n        value = torch.log_softmax(logits[0, :length].detach().float(), dim=-1).cpu()\n        captured.append(value)\n        return output\n\n    pipe.model.forward = spy\n    started = time.monotonic()\n    try:\n        with torch.inference_mode():\n            result = pipe.transcribe([str(audio_path)], lang=[omni_code], batch_size=1)\n    finally:\n        pipe.model.forward = original_forward\n    elapsed = time.monotonic() - started\n    assert len(captured) == 1\n    logits = captured[0].numpy()\n    assert logits.ndim == 2 and logits.shape[1] == len(labels)\n    assert torch.isfinite(captured[0]).all().item()\n    output_hash = hashlib.sha256(str(result).encode("utf-8")).hexdigest()\n    del result, captured\n    gc.collect()\n    return logits, {\n        "seconds": elapsed,\n        "frames": int(logits.shape[0]),\n        "vocabulary": int(logits.shape[1]),\n        "output_sha256": output_hash,\n        "adapter_enabled": bool(enabled),\n    }\n\n\nprobe_records = {}\nfor probe_name in ("base", "lora_mas"):\n    probe = SPEC["probes"][probe_name]\n    audio_path = Path(probe["local_audio_path"]).resolve()\n    assert sha256_file(audio_path) == probe["audio_sha256"]\n    emit("acoustic_probe_start", probe=probe_name)\n    logits, record = capture_logits(\n        audio_path,\n        str(probe["omni_language"]),\n        enabled=(probe_name == "lora_mas"),\n    )\n    probe_records[probe_name] = record\n    if probe_name == "base":\n        base_logits = logits\n    else:\n        lora_logits = logits\n    emit("acoustic_probe_complete", probe=probe_name, seconds=record["seconds"])\n\nassert "base_logits" in globals() and "lora_logits" in globals()\n\ndecoder_records = []\nresident_decoders = 0\nmax_resident_decoders = 0\nfor index, group in enumerate(SPEC["runtime_groups"], 1):\n    config = RUNTIME_CONFIG[group]\n    binary = Path(config["lm_bin"]).resolve()\n    unigrams_path = Path(config["uni_file"]).resolve()\n    assert binary.is_file() and unigrams_path.is_file()\n    assert sha256_file(binary) == config["binary_sha256"]\n    assert sha256_file(unigrams_path) == config["unigrams_sha256"]\n    unigram_list = unigrams_path.read_text(encoding="utf-8").splitlines()\n    assert unigram_list\n    emit("decoder_start", group=group, index=index)\n    started = time.monotonic()\n    decoder = build_ctcdecoder(\n        labels,\n        str(binary),\n        unigrams=unigram_list,\n        alpha=float(config["alpha"]),\n        beta=float(config["beta"]),\n    )\n    resident_decoders += 1\n    max_resident_decoders = max(max_resident_decoders, resident_decoders)\n    assert resident_decoders == 1\n    logits = lora_logits if group == "mas|unscripted" else base_logits\n    hypothesis = decoder.decode(logits, beam_width=int(config["beam"]))\n    elapsed = time.monotonic() - started\n    hypothesis_sha256 = hashlib.sha256(str(hypothesis).encode("utf-8")).hexdigest()\n    decoder_records.append(\n        {\n            "group": group,\n            "asset_id": config["asset_id"],\n            "binary_sha256": config["binary_sha256"],\n            "unigrams_sha256": config["unigrams_sha256"],\n            "alpha": float(config["alpha"]),\n            "beta": float(config["beta"]),\n            "beam": int(config["beam"]),\n            "seconds": elapsed,\n            "hypothesis_sha256": hypothesis_sha256,\n            "hypothesis_words": len(str(hypothesis).split()),\n            "adapter_enabled": group == "mas|unscripted",\n        }\n    )\n    del hypothesis, decoder, unigram_list\n    resident_decoders -= 1\n    gc.collect()\n    assert resident_decoders == 0\n    emit("decoder_complete", group=group, index=index, seconds=elapsed)\n\nassert len(decoder_records) == len(SPEC["runtime_groups"])\nassert max_resident_decoders == 1\n\nRESULT = {\n    "schema": 1,\n    "cell": "20.K5-worker",\n    "status": "PASS",\n    "cpu_only": True,\n    "cuda_available": False,\n    "batch_size": 1,\n    "base_parameters": base_parameters_loaded,\n    "adapter_parameters": copied,\n    "total_neural_parameters": TOTAL_PARAMETERS,\n    "base_load_seconds": base_load_seconds,\n    "lora_wrappers": len(wrappers),\n    "labels": len(labels),\n    "labels_sha256": labels_sha256,\n    "acoustic_probes": probe_records,\n    "decoder_routes": decoder_records,\n    "decoder_routes_completed": len(decoder_records),\n    "max_resident_decoders": max_resident_decoders,\n    "raw_audio_written": False,\n    "raw_text_written": False,\n}\nwrite_json(RESULT, RESULT_PATH)\nemit("worker_complete", decoder_routes=len(decoder_records))\n'

"""20.K5 - audit edge/RSS du candidat KenLM V6 issu de 20.K4.

Cette cellule est volontairement conservative : elle fige les preuves K4,
construit un package candidat sans activer le runtime, puis lance un worker CPU
propre. Le worker charge une seule base BF16, le LoRA Maasai et exactement un
decodeur KenLM a la fois. Aucun audio test n'est lu.
"""

import hashlib
import importlib.metadata
import json
import math
import os
import shutil
import subprocess
import sys
import time
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import psutil
import pyarrow.parquet as pq
import soundfile as sf


# ---------------------------------------------------------------------------
# 1. Contrat fige
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/afrivoices_project")
FT_ROOT = PROJECT_ROOT / "finetune_runs/experiment_run"
REPORT_ROOT = FT_ROOT / "reports"
K4_REPORT_ROOT = REPORT_ROOT / "kenlm_v6_20_K4"
K5_REPORT_ROOT = REPORT_ROOT / "kenlm_v6_20_K5"
PACKAGE_ROOT = FT_ROOT / "final_kenlm_v6_edge_candidate_v1"

EXPECTED_K4_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K4_CONTRACT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K4_REPORT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K4_COMPLETE_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K4_CONFIG_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_ACCEPTED_GROUPS = {
    "sw|unscripted", "kik|unscripted", "mas|scripted"
}

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
RUNTIME_GROUPS = tuple(
    f"{language}|{domain}"
    for language in LANGUAGES
    for domain in ("scripted", "unscripted")
)
LANG_TO_OMNI = {
    "sw": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
    "luo": "luo_Latn", "som": "som_Latn", "mas": "saq_Latn",
}

EDGE_PACKAGE = (
    FT_ROOT
    / "final_lora_edge_candidate_v1/REPLACE_WITH_LOCAL_RUN_ID"
)
BASE_WEIGHT = EDGE_PACKAGE / "base_step1250_bf16.pt"
MAS_ADAPTER = EDGE_PACKAGE / "adapter_mas_step1250.pt"
EDGE_MANIFEST = EDGE_PACKAGE / "edge_manifest_v2.json"
LORA_PLAN = (
    REPORT_ROOT
    / "lora_compatibility_audit_v1/REPLACE_WITH_LOCAL_RUN_ID/lora_plan.parquet"
)
DEV_SELECTION = REPORT_ROOT / "balanced_v4/selected_dev_v4.parquet"
ACTIVE_CONFIG = REPORT_ROOT / "kenlm_tuning_by_domain_v3.json"

BASE_PARAMETERS = 975_675_056
ADAPTER_PARAMETERS = 9_898_496
TOTAL_PARAMETERS = 985_573_552
PARAMETER_LIMIT = 1_000_000_000
RSS_LIMIT_BYTES = 8 * 2**30
RSS_MARGIN = 0.15
RSS_INTERVAL_SECONDS = 0.005
RSS_INTERVAL_P95_LIMIT_SECONDS = 0.050
WORKER_TIMEOUT_SECONDS = 60 * 60
BATCH_SIZE = 1
LORA_PLAN_ROWS = 289

LOCAL_PROBE_ROOT = Path("/content/kenlm_k5_real_dev_probes")
LOCAL_WORK_ROOT = Path("/content/kenlm_k5_worker")
LOCAL_ASSET_ROOT = Path("/content/kenlm_k5_assets")

assert TOTAL_PARAMETERS == BASE_PARAMETERS + ADAPTER_PARAMETERS
assert TOTAL_PARAMETERS < PARAMETER_LIMIT
assert BATCH_SIZE == 1
assert Path("/content/persistent_storage").is_dir(), "Monter Google Drive avant 20.K5."
assert "K5_WORKER_SOURCE" in globals(), "Le worker embarque 20.K5 est absent."


# ---------------------------------------------------------------------------
# 2. Utilitaires
# ---------------------------------------------------------------------------

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path | str, block_size: int = 16 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json(value: Any) -> str:
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str
    )


def canonical_sha256(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_json(path: Path | str) -> dict[str, Any]:
    with open(path, encoding="utf-8") as handle:
        value = json.load(handle)
    assert isinstance(value, dict), path
    return value


def atomic_json(value: Any, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame: pd.DataFrame, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)
    print("csv  ->", path)


def atomic_parquet(frame: pd.DataFrame, path: Path | str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)
    print("parquet ->", path)


def require_child(path: Path | str, root: Path | str) -> Path:
    path = Path(path).resolve()
    root = Path(root).resolve()
    assert path == root or root in path.parents, (path, root)
    return path


def artifact_metadata(path: Path | str) -> dict[str, Any]:
    path = Path(path)
    return {"bytes": path.stat().st_size, "sha256": sha256_file(path)}


def validate_artifacts(root: Path, artifacts: dict[str, Any]) -> None:
    for relative, metadata in artifacts.items():
        path = require_child(root / relative, root)
        assert path.is_file(), path
        assert path.stat().st_size == int(metadata["bytes"]), (path, "size")
        assert sha256_file(path) == metadata["sha256"], (path, "sha256")


def copy_atomic_verified(source: Path, target: Path, expected_sha256: str) -> None:
    assert source.is_file(), source
    assert sha256_file(source) == expected_sha256, source
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and sha256_file(target) == expected_sha256:
        return
    temporary = target.with_name(target.name + f".tmp-{os.getpid()}")
    shutil.copy2(source, temporary)
    assert sha256_file(temporary) == expected_sha256
    os.replace(temporary, target)


def package_versions() -> dict[str, str]:
    result = {}
    for name in (
        "torch", "fairseq2", "omnilingual-asr", "pandas", "pyarrow",
        "soundfile", "pyctcdecode", "kenlm", "psutil",
    ):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "UNKNOWN"
    return result


def canonical_domain(record: dict[str, Any]) -> str:
    if str(record["language"]) == "sw":
        return "unscripted"
    method = str(record.get("speech_method", "")).lower().replace("_", "-")
    if "unscript" in method or "spont" in method:
        return "unscripted"
    if "script" in method or "read" in method:
        return "scripted"
    role = str(record.get("adaptation_role", "")).lower()
    if role == "new_in_domain":
        return "unscripted"
    if role == "legacy_replay":
        return "scripted"
    raise AssertionError((record.get("sample_key"), method, role))


def resolve_unigrams(config: dict[str, Any], language: str) -> Path:
    candidates = []
    for key in ("uni_file", "unigrams", "unigram_file", "unigrams_file"):
        if config.get(key):
            candidates.append(Path(str(config[key])))
    binary = Path(str(config["lm_bin"]))
    candidates.extend(
        [binary.with_name(f"{language}_unigrams.txt"), binary.with_name("unigrams.txt")]
    )
    path = next((candidate for candidate in candidates if candidate.is_file()), None)
    assert path is not None, (language, config)
    return path.resolve()


def manifest_contains_sha(value: Any, expected: str) -> bool:
    if isinstance(value, dict):
        return any(manifest_contains_sha(item, expected) for item in value.values())
    if isinstance(value, list):
        return any(manifest_contains_sha(item, expected) for item in value)
    return isinstance(value, str) and value.lower() == expected.lower()


def descendant_rss_bytes(process: psutil.Process) -> int:
    total = 0
    processes = [process]
    try:
        processes.extend(process.children(recursive=True))
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
    for child in processes:
        try:
            total += int(child.memory_info().rss)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    return total


# ---------------------------------------------------------------------------
# 3. Validation byte-exacte de 20.K4 et du runtime acoustique
# ---------------------------------------------------------------------------

K4_LATEST_PATH = K4_REPORT_ROOT / "LATEST_PASS.json"
K4_LATEST = read_json(K4_LATEST_PATH)
assert K4_LATEST == {
    **K4_LATEST,
    "run_id": EXPECTED_K4_RUN_ID,
    "contract_sha256": EXPECTED_K4_CONTRACT_SHA256,
}
assert K4_LATEST["cell"] == "20.K4"
assert K4_LATEST["status"] == "PASS_READY_FOR_20_K5_EDGE_AUDIT"
assert K4_LATEST["report_sha256"] == EXPECTED_K4_REPORT_SHA256
assert K4_LATEST["complete_sha256"] == EXPECTED_K4_COMPLETE_SHA256
assert K4_LATEST["candidate_config_sha256"] == EXPECTED_K4_CONFIG_SHA256
assert K4_LATEST.get("active_runtime_modified") is False
assert K4_LATEST.get("submission_created") is False

K4_REPORT_PATH = require_child(K4_LATEST["report"], K4_REPORT_ROOT)
K4_COMPLETE_PATH = require_child(K4_LATEST["complete"], K4_REPORT_ROOT)
K4_CONFIG_PATH = require_child(K4_LATEST["candidate_config"], K4_REPORT_ROOT)
assert sha256_file(K4_REPORT_PATH) == EXPECTED_K4_REPORT_SHA256
assert sha256_file(K4_COMPLETE_PATH) == EXPECTED_K4_COMPLETE_SHA256
assert sha256_file(K4_CONFIG_PATH) == EXPECTED_K4_CONFIG_SHA256

K4_REPORT = read_json(K4_REPORT_PATH)
K4_COMPLETE = read_json(K4_COMPLETE_PATH)
K4_CONFIG = read_json(K4_CONFIG_PATH)
K4_RUN_ROOT = K4_REPORT_PATH.parent.resolve()
K4_CONTRACT_DOCUMENT = read_json(K4_RUN_ROOT / "contract.json")
assert K4_CONTRACT_DOCUMENT["contract_sha256"] == EXPECTED_K4_CONTRACT_SHA256
K4_CONTRACT = K4_CONTRACT_DOCUMENT["contract"]
assert K4_REPORT["run_id"] == EXPECTED_K4_RUN_ID
assert K4_REPORT["contract_sha256"] == EXPECTED_K4_CONTRACT_SHA256
assert set(K4_REPORT["accepted_groups"]) == EXPECTED_ACCEPTED_GROUPS
assert K4_REPORT.get("test_audio_read") is False
assert K4_REPORT.get("active_runtime_modified") is False
assert K4_COMPLETE["run_id"] == EXPECTED_K4_RUN_ID
assert K4_COMPLETE["contract_sha256"] == EXPECTED_K4_CONTRACT_SHA256
assert set(K4_COMPLETE["accepted_groups"]) == EXPECTED_ACCEPTED_GROUPS
validate_artifacts(K4_RUN_ROOT, K4_COMPLETE["artifacts"])
assert set(K4_CONFIG) == set(RUNTIME_GROUPS), sorted(K4_CONFIG)

for required in (BASE_WEIGHT, MAS_ADAPTER, EDGE_MANIFEST, LORA_PLAN, DEV_SELECTION):
    assert required.is_file(), required
BASE_SHA = sha256_file(BASE_WEIGHT)
ADAPTER_SHA = sha256_file(MAS_ADAPTER)
EDGE_MANIFEST_SHA = sha256_file(EDGE_MANIFEST)
PLAN_SHA = sha256_file(LORA_PLAN)
DEV_SHA = sha256_file(DEV_SELECTION)
EDGE_MANIFEST_VALUE = read_json(EDGE_MANIFEST)
for expected in (BASE_SHA, ADAPTER_SHA):
    assert manifest_contains_sha(EDGE_MANIFEST_VALUE, expected), expected

EDGE_AUDIT_PATH = REPORT_ROOT / "lora_edge_bf16_audit_v2.json"
EDGE_AUDIT = read_json(EDGE_AUDIT_PATH)
assert EDGE_AUDIT.get("status") == "PASS_READY_FOR_TEST_INFERENCE", EDGE_AUDIT
EDGE_AUDIT_SHA = sha256_file(EDGE_AUDIT_PATH)

# Chaine de provenance : les actifs relus par K5 doivent etre exactement ceux
# qui ont servi a K4, pas seulement des fichiers portant les memes noms.
assert K4_CONTRACT["base_weight_sha256"] == BASE_SHA
assert K4_CONTRACT["mas_adapter_sha256"] == ADAPTER_SHA
assert K4_CONTRACT["lora_plan_sha256"] == PLAN_SHA
assert K4_CONTRACT["dev_selection_sha256"] == DEV_SHA

plan = pd.read_parquet(LORA_PLAN)
assert len(plan) == LORA_PLAN_ROWS
assert int(plan["parameters_per_adapter"].sum()) == ADAPTER_PARAMETERS

assert ACTIVE_CONFIG.is_file(), ACTIVE_CONFIG
ACTIVE_CONFIG_SHA_BEFORE = sha256_file(ACTIVE_CONFIG)

print("=== PREFLIGHT 20.K5 ===")
print("20.K4       :", EXPECTED_K4_RUN_ID)
print("Base BF16   :", BASE_SHA[:16])
print("LoRA Maasai :", ADAPTER_SHA[:16])
print("Parametres  :", f"{TOTAL_PARAMETERS:,}")
print("GPU requis  : non")


# ---------------------------------------------------------------------------
# 4. Inventaire LM et deux sondes acoustiques DEV reelles
# ---------------------------------------------------------------------------

source_assets: OrderedDict[tuple[str, str], dict[str, Any]] = OrderedDict()
route_inventory = []
for group in RUNTIME_GROUPS:
    language, domain = group.split("|")
    config = K4_CONFIG[group]
    binary = require_child(config["lm_bin"], PROJECT_ROOT)
    unigrams = require_child(resolve_unigrams(config, language), PROJECT_ROOT)
    assert "kaggle_test" not in str(binary).lower(), binary
    assert "kaggle_test" not in str(unigrams).lower(), unigrams
    binary_sha = sha256_file(binary)
    unigrams_sha = sha256_file(unigrams)
    content_key = (binary_sha, unigrams_sha)
    if content_key not in source_assets:
        asset_id = f"lm_{len(source_assets):02d}_{binary_sha[:12]}"
        source_assets[content_key] = {
            "asset_id": asset_id,
            "binary": str(binary),
            "binary_sha256": binary_sha,
            "binary_bytes": binary.stat().st_size,
            "unigrams": str(unigrams),
            "unigrams_sha256": unigrams_sha,
            "unigrams_bytes": unigrams.stat().st_size,
        }
    asset = source_assets[content_key]
    route_inventory.append(
        {
            "group": group,
            "language": language,
            "domain": domain,
            "asset_id": asset["asset_id"],
            "binary_sha256": binary_sha,
            "unigrams_sha256": unigrams_sha,
            "alpha": float(config["alpha"]),
            "beta": float(config["beta"]),
            "beam": int(config["beam"]),
        }
    )

assert len(route_inventory) == 12
assert all(row["beam"] in (100, 250) for row in route_inventory)

dev = pd.read_parquet(DEV_SELECTION)
required_columns = {
    "sample_key", "language", "split", "speech_method", "adaptation_role",
    "source_kind", "audio_ref", "parquet_path", "row_index", "seconds",
}
assert required_columns <= set(dev), required_columns - set(dev)
assert dev["sample_key"].astype(str).is_unique
assert dev["split"].astype(str).eq("dev").all()
dev["seconds"] = pd.to_numeric(dev["seconds"], errors="raise")
assert dev["seconds"].between(0.5, 38.0).all()
dev["domain"] = [canonical_domain(row) for row in dev.to_dict("records")]
dev["group"] = dev["language"].astype(str) + "|" + dev["domain"]


def choose_probe(group: str) -> dict[str, Any]:
    frame = dev[dev.group.eq(group)].copy()
    assert not frame.empty, group
    frame["rank"] = frame["sample_key"].astype(str).map(
        lambda value: hashlib.sha256(f"20.K5|{group}|{value}".encode()).hexdigest()
    )
    frame = frame.sort_values(["seconds", "rank"], ascending=[False, True])
    return frame.iloc[0].to_dict()


def materialize_probe(row: dict[str, Any], probe_name: str) -> dict[str, Any]:
    sample_hash = hashlib.sha256(str(row["sample_key"]).encode()).hexdigest()
    destination = LOCAL_PROBE_ROOT / DEV_SHA[:12] / f"{probe_name}_{sample_hash[:20]}.flac"
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = None
    source = None
    if str(row["source_kind"]) == "file":
        source = Path(str(row["audio_ref"])).resolve()
        assert source.is_file(), source
        assert "kaggle_test" not in str(source).lower(), source
        expected_audio_sha = sha256_file(source)
    else:
        parquet_path = Path(str(row["parquet_path"])).resolve()
        assert parquet_path.is_file(), parquet_path
        assert "kaggle_test" not in str(parquet_path).lower(), parquet_path
        table = pq.read_table(parquet_path, columns=["audio_bytes"])
        payload = table.column("audio_bytes")[int(row["row_index"])].as_py()
        assert payload
        expected_audio_sha = hashlib.sha256(bytes(payload)).hexdigest()
    if not destination.is_file() or sha256_file(destination) != expected_audio_sha:
        temporary = destination.with_name(destination.name + f".tmp-{os.getpid()}")
        if source is not None:
            shutil.copyfile(source, temporary)
        else:
            with open(temporary, "wb") as handle:
                handle.write(bytes(payload))
        assert sha256_file(temporary) == expected_audio_sha
        os.replace(temporary, destination)
    info = sf.info(destination)
    assert info.samplerate == 16_000 and info.frames > 0, (destination, info)
    assert 0.5 <= info.duration <= 38.1, (destination, info.duration)
    return {
        "group": row["group"],
        "language": row["language"],
        "omni_language": LANG_TO_OMNI[str(row["language"])],
        "sample_key_sha256": sample_hash,
        "audio_sha256": expected_audio_sha,
        "bytes": destination.stat().st_size,
        "frames": int(info.frames),
        "samplerate": int(info.samplerate),
        "seconds": float(info.duration),
        "local_audio_path": str(destination),
    }


PROBES = {
    "base": materialize_probe(choose_probe("kik|scripted"), "base"),
    "lora_mas": materialize_probe(choose_probe("mas|unscripted"), "lora_mas"),
}

# La version publique du manifest ne contient aucun chemin audio local.
PROBE_PUBLIC = {
    name: {key: value for key, value in record.items() if key != "local_audio_path"}
    for name, record in PROBES.items()
}


# ---------------------------------------------------------------------------
# 5. Contrat, package candidat et reprise sure
# ---------------------------------------------------------------------------

ENVIRONMENT = {
    "created_at": now_iso(),
    "python": sys.version,
    "packages": package_versions(),
    "system_ram_bytes": int(psutil.virtual_memory().total),
    "cpu_count": int(psutil.cpu_count() or 1),
    "cuda_visible_devices_parent": os.environ.get("CUDA_VISIBLE_DEVICES"),
}
SOFTWARE_FINGERPRINT = {
    "python": sys.version.split()[0],
    "packages": package_versions(),
    "omnilingual_asr_git_commit": K4_CONTRACT["software_fingerprint"][
        "omnilingual_asr_git_commit"
    ],
}

CONTRACT = {
    "schema": 1,
    "cell": "20.K5",
    "purpose": "edge_rss_audit_before_test_inference",
    "k4": {
        "run_id": EXPECTED_K4_RUN_ID,
        "contract_sha256": EXPECTED_K4_CONTRACT_SHA256,
        "report_sha256": EXPECTED_K4_REPORT_SHA256,
        "complete_sha256": EXPECTED_K4_COMPLETE_SHA256,
        "candidate_config_sha256": EXPECTED_K4_CONFIG_SHA256,
        "accepted_groups": sorted(EXPECTED_ACCEPTED_GROUPS),
    },
    "runtime": {
        "base_sha256": BASE_SHA,
        "adapter_sha256": ADAPTER_SHA,
        "edge_manifest_sha256": EDGE_MANIFEST_SHA,
        "edge_audit_sha256": EDGE_AUDIT_SHA,
        "lora_plan_sha256": PLAN_SHA,
        "dev_selection_sha256": DEV_SHA,
        "base_parameters": BASE_PARAMETERS,
        "adapter_parameters": ADAPTER_PARAMETERS,
        "total_parameters": TOTAL_PARAMETERS,
        "batch_size": BATCH_SIZE,
        "cpu_worker": True,
    },
    "edge_gate": {
        "rss_limit_bytes": RSS_LIMIT_BYTES,
        "safety_margin": RSS_MARGIN,
        "sample_interval_seconds": RSS_INTERVAL_SECONDS,
        "max_resident_decoders": 1,
    },
    "routes": route_inventory,
    "source_assets": [
        {key: value for key, value in asset.items() if key not in ("binary", "unigrams")}
        for asset in source_assets.values()
    ],
    "probes": PROBE_PUBLIC,
    "software_fingerprint": SOFTWARE_FINGERPRINT,
    "software_fingerprint_sha256": canonical_sha256(SOFTWARE_FINGERPRINT),
    "worker_sha256": hashlib.sha256(K5_WORKER_SOURCE.encode("utf-8")).hexdigest(),
    "active_config_sha256_before": ACTIVE_CONFIG_SHA_BEFORE,
    "forbidden_actions": [
        "test_audio_read", "active_config_replace", "submission_create",
        "raw_reference_or_hypothesis_write",
    ],
}
CONTRACT_SHA = canonical_sha256(CONTRACT)
RUN_ID = CONTRACT_SHA[:16]
REPORT_STAGING = K5_REPORT_ROOT / f"{RUN_ID}.staging"
REPORT_FINAL = K5_REPORT_ROOT / RUN_ID
PACKAGE_STAGING = PACKAGE_ROOT / f"{RUN_ID}.staging"
PACKAGE_FINAL = PACKAGE_ROOT / RUN_ID


def validate_completed_report(root: Path) -> dict[str, Any] | None:
    complete_path = root / "_COMPLETE.json"
    report_path = root / "kenlm_edge_audit_20_K5.json"
    if not complete_path.is_file() or not report_path.is_file():
        return None
    complete = read_json(complete_path)
    if complete.get("run_id") != RUN_ID or complete.get("contract_sha256") != CONTRACT_SHA:
        return None
    assert complete.get("cell") == "20.K5"
    assert complete.get("status") in {
        "PASS_READY_FOR_20_K6_TEST_INFERENCE", "BLOCKED_EDGE_RAM", "FAIL_RUNTIME_AUDIT"
    }
    assert complete.get("test_audio_read") is False
    assert complete.get("active_runtime_modified") is False
    assert complete.get("submission_created") is False
    validate_artifacts(root, complete["artifacts"])
    report = read_json(report_path)
    assert report["run_id"] == RUN_ID
    assert report["contract_sha256"] == CONTRACT_SHA
    assert report["status"] == complete["status"]
    assert report.get("test_audio_read") is False
    assert report.get("active_runtime_modified") is False
    assert report.get("submission_created") is False
    assert Path(report["package"]).resolve() == PACKAGE_FINAL.resolve()
    package = validate_package(PACKAGE_FINAL)
    assert package is not None
    assert sha256_file(PACKAGE_FINAL / "package_manifest.json") == report[
        "package_manifest_sha256"
    ]
    if report["status"] == "PASS_READY_FOR_20_K6_TEST_INFERENCE":
        assert all(report["runtime_checks"].values())
    return report


def validate_package(root: Path) -> dict[str, Any] | None:
    manifest_path = root / "package_manifest.json"
    if not manifest_path.is_file():
        return None
    manifest = read_json(manifest_path)
    if manifest.get("run_id") != RUN_ID or manifest.get("contract_sha256") != CONTRACT_SHA:
        return None
    validate_artifacts(root, manifest["artifacts"])
    assert manifest.get("path_mode") == "relative_to_package_root"
    packaged_config = read_json(root / "runtime_config.json")
    assert set(packaged_config) == set(RUNTIME_GROUPS)
    for group, config in packaged_config.items():
        binary = require_child(root / config["lm_bin"], root)
        unigrams = require_child(root / config["uni_file"], root)
        assert binary.is_file() and sha256_file(binary) == config["binary_sha256"]
        assert unigrams.is_file() and sha256_file(unigrams) == config["unigrams_sha256"]
        assert int(config["beam"]) in (100, 250)
    expected_external = {
        "base_weight": (BASE_WEIGHT, BASE_SHA),
        "adapter_weight": (MAS_ADAPTER, ADAPTER_SHA),
        "lora_plan": (LORA_PLAN, PLAN_SHA),
    }
    for role, (expected_path, expected_sha) in expected_external.items():
        proof = manifest["external_immutable_assets"][role]
        assert Path(proof["path"]).resolve() == expected_path.resolve()
        assert expected_path.is_file() and sha256_file(expected_path) == expected_sha
        assert proof["sha256"] == expected_sha
    return manifest


existing = validate_completed_report(REPORT_FINAL) if REPORT_FINAL.is_dir() else None
if existing is not None and existing["status"] != "PASS_READY_FOR_20_K6_TEST_INFERENCE":
    # Conserver la preuve d'un echec/blocage, mais autoriser une nouvelle sonde
    # (un incident CPU/Drive transitoire ne doit pas figer K5 pour toujours).
    attempts = K5_REPORT_ROOT / "attempts"
    attempts.mkdir(parents=True, exist_ok=True)
    archived = attempts / f"{RUN_ID}_{int(time.time())}_{existing['status'].lower()}"
    os.replace(REPORT_FINAL, archived)
    print("[reprise] tentative non-PASS archivee :", archived)
    existing = None
if existing is not None:
    latest_existing = {
        "schema": 1,
        "cell": "20.K5",
        "status": existing["status"],
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "report": str(REPORT_FINAL / "kenlm_edge_audit_20_K5.json"),
        "report_sha256": sha256_file(REPORT_FINAL / "kenlm_edge_audit_20_K5.json"),
        "complete": str(REPORT_FINAL / "_COMPLETE.json"),
        "complete_sha256": sha256_file(REPORT_FINAL / "_COMPLETE.json"),
        "package": str(PACKAGE_FINAL),
        "active_runtime_modified": False,
        "submission_created": False,
    }
    atomic_json(latest_existing, K5_REPORT_ROOT / "LATEST_ATTEMPT.json")
    atomic_json(latest_existing, K5_REPORT_ROOT / "LATEST.json")
    if existing["status"] == "PASS_READY_FOR_20_K6_TEST_INFERENCE":
        atomic_json(latest_existing, K5_REPORT_ROOT / "LATEST_PASS.json")
    print("\n[reprise] 20.K5 deja termine et verifie :", existing["status"])
    print("Rapport :", latest_existing["report"])
else:
    assert not REPORT_FINAL.exists(), REPORT_FINAL
    if REPORT_STAGING.exists():
        shutil.rmtree(REPORT_STAGING)
    REPORT_STAGING.mkdir(parents=True)

    package_manifest = validate_package(PACKAGE_FINAL) if PACKAGE_FINAL.is_dir() else None
    if package_manifest is None:
        assert not PACKAGE_FINAL.exists(), PACKAGE_FINAL
        if PACKAGE_STAGING.exists():
            shutil.rmtree(PACKAGE_STAGING)
        PACKAGE_STAGING.mkdir(parents=True)

        portable_config = json.loads(json.dumps(K4_CONFIG))
        package_artifacts = {}
        for asset in source_assets.values():
            asset_dir = PACKAGE_STAGING / "lms" / asset["asset_id"]
            binary_target = asset_dir / "model.binary"
            unigrams_target = asset_dir / "unigrams.txt"
            copy_atomic_verified(Path(asset["binary"]), binary_target, asset["binary_sha256"])
            copy_atomic_verified(
                Path(asset["unigrams"]), unigrams_target, asset["unigrams_sha256"]
            )
            package_artifacts[str(binary_target.relative_to(PACKAGE_STAGING))] = (
                artifact_metadata(binary_target)
            )
            package_artifacts[str(unigrams_target.relative_to(PACKAGE_STAGING))] = (
                artifact_metadata(unigrams_target)
            )

        asset_by_id = {asset["asset_id"]: asset for asset in source_assets.values()}
        route_by_group = {row["group"]: row for row in route_inventory}
        for group in RUNTIME_GROUPS:
            route = route_by_group[group]
            asset = asset_by_id[route["asset_id"]]
            relative_asset_dir = Path("lms") / asset["asset_id"]
            portable_config[group]["lm_bin"] = str(relative_asset_dir / "model.binary")
            portable_config[group]["uni_file"] = str(relative_asset_dir / "unigrams.txt")
            portable_config[group]["asset_id"] = asset["asset_id"]
            portable_config[group]["binary_sha256"] = asset["binary_sha256"]
            portable_config[group]["unigrams_sha256"] = asset["unigrams_sha256"]
            portable_config[group]["alpha"] = route["alpha"]
            portable_config[group]["beta"] = route["beta"]
            portable_config[group]["beam"] = route["beam"]

        portable_path = PACKAGE_STAGING / "runtime_config.json"
        atomic_json(portable_config, portable_path)
        package_artifacts["runtime_config.json"] = artifact_metadata(portable_path)
        package_manifest = {
            "schema": 1,
            "status": "CANDIDATE_NOT_ACTIVE",
            "created_at": now_iso(),
            "run_id": RUN_ID,
            "contract_sha256": CONTRACT_SHA,
            "k4_candidate_config_sha256": EXPECTED_K4_CONFIG_SHA256,
            "runtime_groups": list(RUNTIME_GROUPS),
            "unique_lm_assets": len(source_assets),
            "path_mode": "relative_to_package_root",
            "artifacts": package_artifacts,
            "external_immutable_assets": {
                "base_weight": {"path": str(BASE_WEIGHT), "sha256": BASE_SHA},
                "adapter_weight": {"path": str(MAS_ADAPTER), "sha256": ADAPTER_SHA},
                "lora_plan": {"path": str(LORA_PLAN), "sha256": PLAN_SHA},
            },
            "active_runtime_modified": False,
        }
        atomic_json(package_manifest, PACKAGE_STAGING / "package_manifest.json")
        assert validate_package(PACKAGE_STAGING) is not None
        os.replace(PACKAGE_STAGING, PACKAGE_FINAL)
        package_manifest = validate_package(PACKAGE_FINAL)
        assert package_manifest is not None
    else:
        print("[reprise] package candidat deja verifie :", PACKAGE_FINAL)

    runtime_config_path = PACKAGE_FINAL / "runtime_config.json"
    runtime_config = read_json(runtime_config_path)
    assert set(runtime_config) == set(RUNTIME_GROUPS)

    # L'audit edge ne mmappe pas les gros fichiers via Drive/FUSE : on copie
    # une fois les memes octets sur le disque local, puis on re-verifie les SHA.
    assert shutil.disk_usage("/content").free >= 6 * 2**30, (
        "Au moins 6 Gio libres sont requis pour la sonde edge locale."
    )
    LOCAL_ASSET_ROOT.mkdir(parents=True, exist_ok=True)
    local_base = LOCAL_ASSET_ROOT / BASE_SHA[:16] / "base_step1250_bf16.pt"
    local_adapter = LOCAL_ASSET_ROOT / ADAPTER_SHA[:16] / "adapter_mas_step1250.pt"
    local_plan = LOCAL_ASSET_ROOT / PLAN_SHA[:16] / "lora_plan.parquet"
    copy_atomic_verified(BASE_WEIGHT, local_base, BASE_SHA)
    copy_atomic_verified(MAS_ADAPTER, local_adapter, ADAPTER_SHA)
    copy_atomic_verified(LORA_PLAN, local_plan, PLAN_SHA)

    local_runtime_config = json.loads(json.dumps(runtime_config))
    for group, config in local_runtime_config.items():
        source_binary = PACKAGE_FINAL / config["lm_bin"]
        source_unigrams = PACKAGE_FINAL / config["uni_file"]
        binary_sha = config["binary_sha256"]
        unigrams_sha = config["unigrams_sha256"]
        local_dir = LOCAL_ASSET_ROOT / config["asset_id"]
        local_binary = local_dir / "model.binary"
        local_unigrams = local_dir / "unigrams.txt"
        copy_atomic_verified(source_binary, local_binary, binary_sha)
        copy_atomic_verified(source_unigrams, local_unigrams, unigrams_sha)
        config["lm_bin"] = str(local_binary)
        config["uni_file"] = str(local_unigrams)
    local_runtime_config_path = LOCAL_WORK_ROOT / f"{RUN_ID}_runtime_config.json"
    LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_json(local_runtime_config, local_runtime_config_path)

    atomic_json(CONTRACT, REPORT_STAGING / "contract.json")
    atomic_json(ENVIRONMENT, REPORT_STAGING / "environment.json")
    atomic_json(PROBE_PUBLIC, REPORT_STAGING / "probe_manifest.json")
    atomic_csv(pd.DataFrame(route_inventory), REPORT_STAGING / "route_inventory.csv")
    atomic_json(K4_CONFIG, REPORT_STAGING / "approved_k4_candidate_config.json")
    worker_path = REPORT_STAGING / "edge_worker.py"
    worker_path.write_text(K5_WORKER_SOURCE, encoding="utf-8")
    assert sha256_file(worker_path) == CONTRACT["worker_sha256"]

    worker_input = LOCAL_WORK_ROOT / f"{RUN_ID}_input.json"
    worker_result_local = LOCAL_WORK_ROOT / f"{RUN_ID}_result.json"
    worker_log = REPORT_STAGING / "worker.log"
    if worker_result_local.exists():
        worker_result_local.unlink()
    spec = {
        "base_parameters": BASE_PARAMETERS,
        "adapter_parameters": ADAPTER_PARAMETERS,
        "total_neural_parameters": TOTAL_PARAMETERS,
        "base_weight": str(local_base),
        "base_weight_sha256": BASE_SHA,
        "adapter_weight": str(local_adapter),
        "adapter_weight_sha256": ADAPTER_SHA,
        "lora_plan": str(local_plan),
        "lora_plan_sha256": PLAN_SHA,
        "lora_plan_rows": LORA_PLAN_ROWS,
        "runtime_config": str(local_runtime_config_path),
        "runtime_config_sha256": sha256_file(local_runtime_config_path),
        "runtime_groups": list(RUNTIME_GROUPS),
        "batch_size": 1,
        "edge_card_name": f"afrivoices_k5_edge_{RUN_ID[:12]}",
        "probes": PROBES,
    }
    atomic_json(spec, worker_input)

    print("\nSonde edge propre : CPU, 12 routes, deux audios DEV reels...")
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = ""
    started = time.monotonic()
    trace = []
    timed_out = False
    last_heartbeat = started
    with open(worker_log, "w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            [sys.executable, str(worker_path), str(worker_input), str(worker_result_local)],
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            env=environment,
            cwd=str(REPORT_STAGING),
        )
        monitored = psutil.Process(process.pid)
        while process.poll() is None:
            now = time.monotonic()
            current_rss = descendant_rss_bytes(monitored)
            trace.append(
                {
                    "monotonic": now,
                    "elapsed_seconds": now - started,
                    "rss_bytes": current_rss,
                }
            )
            if now - last_heartbeat >= 60:
                print(
                    f"  worker actif | {((now-started)/60):.1f} min | "
                    f"RSS arbre={current_rss/2**30:.3f} Gio",
                    flush=True,
                )
                last_heartbeat = now
            if now - started > WORKER_TIMEOUT_SECONDS:
                timed_out = True
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=15)
                break
            time.sleep(RSS_INTERVAL_SECONDS)
        returncode = int(process.wait())
        trace.append(
            {
                "monotonic": time.monotonic(),
                "elapsed_seconds": time.monotonic() - started,
                "rss_bytes": 0,
            }
        )

    trace_frame = pd.DataFrame(trace)
    atomic_parquet(trace_frame, REPORT_STAGING / "rss_trace.parquet")
    peak_rss = int(trace_frame.rss_bytes.max()) if len(trace_frame) else 0
    adjusted_peak = int(math.ceil(peak_rss * (1.0 + RSS_MARGIN)))
    intervals = trace_frame.monotonic.diff().dropna()
    observed_interval_p95 = float(intervals.quantile(0.95)) if len(intervals) else None
    observed_interval_max = float(intervals.max()) if len(intervals) else None
    sampling_ok = (
        len(trace_frame) >= 100
        and peak_rss > 0
        and observed_interval_p95 is not None
        and observed_interval_p95 <= RSS_INTERVAL_P95_LIMIT_SECONDS
    )
    worker_ok = (not timed_out) and returncode == 0 and worker_result_local.is_file()
    worker_result = read_json(worker_result_local) if worker_ok else {}
    if worker_ok:
        shutil.copy2(worker_result_local, REPORT_STAGING / "worker_result.json")
    else:
        atomic_json(
            {"status": "FAIL", "returncode": returncode},
            REPORT_STAGING / "worker_result.json",
        )

    returned_routes = worker_result.get("decoder_routes", [])
    returned_groups = [row.get("group") for row in returned_routes]
    returned_probes = worker_result.get("acoustic_probes", {})
    route_flags_ok = all(
        bool(row.get("adapter_enabled")) == (row.get("group") == "mas|unscripted")
        for row in returned_routes
    )
    probe_flags_ok = (
        set(returned_probes) == {"base", "lora_mas"}
        and returned_probes.get("base", {}).get("adapter_enabled") is False
        and returned_probes.get("lora_mas", {}).get("adapter_enabled") is True
    )
    runtime_checks = {
        "worker_returncode_zero": returncode == 0,
        "worker_not_timed_out": not timed_out,
        "worker_status_pass": worker_result.get("status") == "PASS",
        "cpu_only": worker_result.get("cpu_only") is True,
        "batch_size_one": worker_result.get("batch_size") == 1,
        "parameters_exact": worker_result.get("total_neural_parameters") == TOTAL_PARAMETERS,
        "two_real_dev_probes": probe_flags_ok,
        "twelve_routes": (
            worker_result.get("decoder_routes_completed") == 12
            and len(returned_groups) == 12
            and set(returned_groups) == set(RUNTIME_GROUPS)
            and len(set(returned_groups)) == 12
            and route_flags_ok
        ),
        "one_decoder_resident": worker_result.get("max_resident_decoders") == 1,
        "rss_sampling_density_valid": sampling_ok,
        "rss_under_limit_with_margin": adjusted_peak < RSS_LIMIT_BYTES,
    }
    ACTIVE_CONFIG_SHA_AFTER = sha256_file(ACTIVE_CONFIG) if ACTIVE_CONFIG.is_file() else None
    runtime_checks["active_config_unchanged"] = (
        ACTIVE_CONFIG.is_file() and ACTIVE_CONFIG_SHA_AFTER == ACTIVE_CONFIG_SHA_BEFORE
    )

    status = (
        "PASS_READY_FOR_20_K6_TEST_INFERENCE"
        if all(runtime_checks.values())
        else (
            "BLOCKED_EDGE_RAM"
            if worker_ok and adjusted_peak >= RSS_LIMIT_BYTES
            else "FAIL_RUNTIME_AUDIT"
        )
    )

    route_seconds = [
        float(row["seconds"]) for row in worker_result.get("decoder_routes", [])
    ]
    SUMMARY = pd.DataFrame(
        [
            {
                "status": status,
                "worker_returncode": returncode,
                "worker_timed_out": timed_out,
                "elapsed_seconds": time.monotonic() - started,
                "peak_rss_gib": peak_rss / 2**30,
                "peak_plus_15pct_gib": adjusted_peak / 2**30,
                "rss_limit_gib": RSS_LIMIT_BYTES / 2**30,
                "rss_interval_p95_seconds": observed_interval_p95,
                "rss_interval_max_seconds": observed_interval_max,
                "routes": worker_result.get("decoder_routes_completed", 0),
                "max_resident_decoders": worker_result.get("max_resident_decoders"),
                "decoder_seconds_median": (
                    float(pd.Series(route_seconds).median()) if route_seconds else None
                ),
                "decoder_seconds_p95": (
                    float(pd.Series(route_seconds).quantile(0.95)) if route_seconds else None
                ),
            }
        ]
    )
    atomic_csv(SUMMARY, REPORT_STAGING / "edge_summary.csv")

    REPORT = {
        "schema": 1,
        "cell": "20.K5",
        "status": status,
        "finished_at": now_iso(),
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "k4_run_id": EXPECTED_K4_RUN_ID,
        "k4_candidate_config_sha256": EXPECTED_K4_CONFIG_SHA256,
        "package": str(PACKAGE_FINAL),
        "package_manifest_sha256": sha256_file(PACKAGE_FINAL / "package_manifest.json"),
        "neural_parameters": TOTAL_PARAMETERS,
        "parameter_limit": PARAMETER_LIMIT,
        "rss": {
            "peak_bytes": peak_rss,
            "peak_gib": peak_rss / 2**30,
            "safety_margin": RSS_MARGIN,
            "adjusted_peak_bytes": adjusted_peak,
            "adjusted_peak_gib": adjusted_peak / 2**30,
            "limit_bytes": RSS_LIMIT_BYTES,
            "limit_gib": 8.0,
            "sample_interval_seconds": RSS_INTERVAL_SECONDS,
            "observed_interval_p95_seconds": observed_interval_p95,
            "observed_interval_max_seconds": observed_interval_max,
            "interval_p95_limit_seconds": RSS_INTERVAL_P95_LIMIT_SECONDS,
            "process_tree_sampled": True,
        },
        "runtime_checks": runtime_checks,
        "worker": worker_result,
        "probes": PROBE_PUBLIC,
        "test_audio_read": False,
        "raw_references_or_hypotheses_written": False,
        "active_runtime_modified": False,
        "submission_created": False,
        "next_step": (
            "20.K6 — inference test avec ce package exact"
            if status == "PASS_READY_FOR_20_K6_TEST_INFERENCE"
            else "Corriger le blocage puis relancer 20.K5 ; ne pas lancer l'inference test"
        ),
    }
    atomic_json(REPORT, REPORT_STAGING / "kenlm_edge_audit_20_K5.json")

    artifact_names = [
        "contract.json", "environment.json", "probe_manifest.json",
        "route_inventory.csv", "approved_k4_candidate_config.json",
        "edge_worker.py", "worker.log", "worker_result.json",
        "rss_trace.parquet", "edge_summary.csv", "kenlm_edge_audit_20_K5.json",
    ]
    artifacts = {
        name: artifact_metadata(REPORT_STAGING / name) for name in artifact_names
    }
    COMPLETE = {
        "schema": 1,
        "cell": "20.K5",
        "status": status,
        "finished_at": now_iso(),
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "artifacts": artifacts,
        "package": str(PACKAGE_FINAL),
        "active_runtime_modified": False,
        "submission_created": False,
        "test_audio_read": False,
    }
    atomic_json(COMPLETE, REPORT_STAGING / "_COMPLETE.json")
    assert validate_completed_report(REPORT_STAGING) is not None
    os.replace(REPORT_STAGING, REPORT_FINAL)
    assert validate_completed_report(REPORT_FINAL) is not None

    LATEST = {
        "schema": 1,
        "cell": "20.K5",
        "status": status,
        "run_id": RUN_ID,
        "contract_sha256": CONTRACT_SHA,
        "report": str(REPORT_FINAL / "kenlm_edge_audit_20_K5.json"),
        "report_sha256": sha256_file(REPORT_FINAL / "kenlm_edge_audit_20_K5.json"),
        "complete": str(REPORT_FINAL / "_COMPLETE.json"),
        "complete_sha256": sha256_file(REPORT_FINAL / "_COMPLETE.json"),
        "package": str(PACKAGE_FINAL),
        "active_runtime_modified": False,
        "submission_created": False,
    }
    atomic_json(LATEST, K5_REPORT_ROOT / "LATEST_ATTEMPT.json")
    atomic_json(LATEST, K5_REPORT_ROOT / "LATEST.json")
    if status == "PASS_READY_FOR_20_K6_TEST_INFERENCE":
        atomic_json(LATEST, K5_REPORT_ROOT / "LATEST_PASS.json")

    print("\n=== AUDIT EDGE 20.K5 ===")
    print(SUMMARY.to_string(index=False))
    print("\nSTATUT 20.K5 :", status)
    print("Package candidat :", PACKAGE_FINAL)
    print("Rapport           :", LATEST["report"])
    print("Runtime actif modifie : non | Soumission creee : non")
    if status != "PASS_READY_FOR_20_K6_TEST_INFERENCE":
        print("NE PAS lancer l'inference test.")


## 20.K5.C — Résultat publié (lecture seule)

In [ ]:
# 20.K5.C — Lecture seule du résultat publié
import json
from pathlib import Path
import pandas as pd

root = Path("/content/afrivoices_project/finetune_runs/experiment_run/reports/kenlm_v6_20_K5")
latest_path = root / "LATEST.json"
assert latest_path.is_file(), "20.K5.B n'est pas encore terminé."
latest = json.load(open(latest_path, encoding="utf-8"))
print(json.dumps(latest, indent=2, ensure_ascii=False))
summary = Path(latest["report"]).parent / "edge_summary.csv"
display(pd.read_csv(summary))
if latest["status"] == "PASS_READY_FOR_20_K6_TEST_INFERENCE":
    print("\n✅ Audit edge PASS. Étape suivante : 20.K6 avec ce package exact.")
else:
    print("\n⛔ Audit non PASS : ne pas lancer l'inférence test.")
